In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input,Bidirectional,Conv1D, MaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import re
import io
import joblib

# --- Configuration ---\n",
DATASET_PATH = '/kaggle/input/mobifall-dataset-v20/MobiFall_Dataset_v2.0'
TARGET_SAMPLING_RATE_HZ = 50.0  # Target sampling rate in Hz
TARGET_SAMPLING_PERIOD = f"{int(1000 / TARGET_SAMPLING_RATE_HZ)}ms"
SEQUENCE_LENGTH = int(TARGET_SAMPLING_RATE_HZ * 4) # 100 samples for 2 seconds at 50Hz
STEP = int(TARGET_SAMPLING_RATE_HZ * 1)          # 50 samples for 1 second step at 50Hz

SENSOR_CODES = ["acc", "gyro", "ori"]
EXPECTED_COLUMNS = {
    "acc": ["acc_x", "acc_y", "acc_z"],
    "gyro": ["gyro_x", "gyro_y", "gyro_z"],
    "ori": ["ori_azimuth", "ori_pitch", "ori_roll"]
}
ALL_FEATURE_COLUMNS = [
    "acc_x", "acc_y", "acc_z", "acc_smv",  # acc_smv added
    "gyro_x", "gyro_y", "gyro_z", "gyro_smv", # gyro_smv added
    "ori_azimuth", "ori_pitch", "ori_roll"
]
from sklearn.utils.class_weight import compute_class_weight


def load_and_resample_sensor_file(filepath, sensor_code):
    """Loads a single sensor file, converts timestamps, and resamples."""
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()

        data_start_line_index = -1
        for i, line in enumerate(lines):
            if line.strip().upper() == "@DATA":
                data_start_line_index = i + 1
                break
        if data_start_line_index == -1 or data_start_line_index >= len(lines):
            return None # No data marker or no data after marker

        data_string = "".join(lines[data_start_line_index:])
        if not data_string.strip():
            return None # No data content

        df = pd.read_csv(io.StringIO(data_string), header=None, usecols=[0, 1, 2, 3])
        if df.empty:
            return None

        df.columns = ['timestamp_ns'] + EXPECTED_COLUMNS[sensor_code]
        df['timestamp'] = pd.to_datetime(df['timestamp_ns'], unit='ns')
        df = df.set_index('timestamp').drop(columns=['timestamp_ns'])
        df = df.sort_index() # Ensure chronological order

       
        df_resampled = df.resample(TARGET_SAMPLING_PERIOD).mean().interpolate(method='linear', limit_direction='both')

        if sensor_code == 'acc':
            # Ensure columns exist before trying to access them
            if all(col in df_resampled.columns for col in ['acc_x', 'acc_y', 'acc_z']):
                df_resampled['acc_smv'] = np.sqrt(
                    df_resampled['acc_x']**2 + df_resampled['acc_y']**2 + df_resampled['acc_z']**2
                )
            else:
                print(f"Warning: Missing acc_x, acc_y, or acc_z columns for SMV in {filepath} after resampling.")
                pass 

        elif sensor_code == 'gyro':
            if all(col in df_resampled.columns for col in ['gyro_x', 'gyro_y', 'gyro_z']):
                df_resampled['gyro_smv'] = np.sqrt(
                    df_resampled['gyro_x']**2 + df_resampled['gyro_y']**2 + df_resampled['gyro_z']**2
                )
            else:
                print(f"Warning: Missing gyro_x, gyro_y, or gyro_z columns for SMV in {filepath} after resampling.")
                pass # Or df_resampled['gyro_smv'] = np.nan
        # No SMV for 'ori' in this typical setup
        
        return df_resampled

    except pd.errors.EmptyDataError:
        # print(f"Debug: pd.errors.EmptyDataError for {filepath}")
        return None
    except ValueError as ve:
        # print(f"Debug: ValueError for {filepath}: {ve}")
        return None
    except Exception as e:
        print(f"Error processing/resampling file {filepath}: {e}. Skipping.")
        return None

def load_data_from_structured_folders(dataset_root_path):
    print(f"Scanning for data in: {dataset_root_path}")
    if not os.path.isdir(dataset_root_path):
        print(f"ERROR: Dataset root path '{dataset_root_path}' not found.")
        return [], []

    trial_sensor_files_map = defaultdict(lambda: defaultdict(str))
    trial_metadata_map = {}
    file_count = 0
    expected_path_depth = 3

    # First pass: Collect all file paths for each trial
    for dirpath, _, filenames in os.walk(dataset_root_path):
        relative_path = os.path.relpath(dirpath, dataset_root_path)
        path_parts = relative_path.split(os.sep)
        if relative_path == '.': continue

        if len(path_parts) == expected_path_depth:
            subject_folder, category_folder, activity_code_folder = path_parts
            subject_match = re.fullmatch(r'sub(\d+)', subject_folder, re.IGNORECASE)
            if not subject_match: continue
            current_subject_id = int(subject_match.group(1))

            if category_folder.upper() not in ["FALLS", "ADL"]: continue
            current_category = category_folder.upper()
            current_activity_code = activity_code_folder.upper()

            for filename in filenames:
                if filename.endswith(".txt"):
                    file_count += 1
                    filepath = os.path.join(dirpath, filename)
                    fname_parts = filename.replace('.txt', '').split('_')
                    if len(fname_parts) != 4:
                        continue

                    _, sensor_code, _, trial_no_str = fname_parts
                    sensor_code = sensor_code.lower()
                    if sensor_code not in SENSOR_CODES: continue

                    try:
                        trial_no = int(trial_no_str)
                    except ValueError:
                        continue

                    trial_key = (current_subject_id, current_activity_code, trial_no)
                    trial_sensor_files_map[trial_key][sensor_code] = filepath
                    if trial_key not in trial_metadata_map:
                         trial_metadata_map[trial_key] = {"category": current_category, "activity_code": current_activity_code}
    print(f"\nScanned {file_count} .txt files.")

    # Second pass: Process each trial
    processed_trials_data = []
    labels = []
    print(f"\nProcessing and combining {len(trial_sensor_files_map)} unique trials...")
    skipped_missing_sensors = 0
    skipped_resampling_issue = 0
    skipped_alignment_issue = 0
    skipped_too_short_after_processing = 0

    for trial_key, sensor_files in trial_sensor_files_map.items():
        if not all(s_code in sensor_files for s_code in SENSOR_CODES):
            skipped_missing_sensors += 1
            continue

        resampled_dfs = {}
        valid_trial = True
        for sensor_code in SENSOR_CODES:
            filepath = sensor_files[sensor_code]
            # print(f"Processing file: {os.path.basename(filepath)} for trial {trial_key}") # More verbose
            df_resampled = load_and_resample_sensor_file(filepath, sensor_code)
            if df_resampled is None or df_resampled.empty:
                # print(f"Warning: Could not resample or got empty df for {filepath}")
                valid_trial = False
                break
            resampled_dfs[sensor_code] = df_resampled
        
        if not valid_trial:
            skipped_resampling_issue += 1
            continue

        # Align based on common time index range
        try:
            common_start_time = max(df.index.min() for df in resampled_dfs.values())
            common_end_time = min(df.index.max() for df in resampled_dfs.values())

            if common_start_time >= common_end_time:
                skipped_alignment_issue +=1
                continue

            aligned_sensor_data = []
            for sensor_code in SENSOR_CODES: # Ensure consistent order
                df = resampled_dfs[sensor_code]
                df_aligned = df[(df.index >= common_start_time) & (df.index <= common_end_time)]
                aligned_sensor_data.append(df_aligned.reset_index(drop=True)) # Reset index for concat

            if not all(len(df) > 0 for df in aligned_sensor_data) or \
               not all(len(df) == len(aligned_sensor_data[0]) for df in aligned_sensor_data): # Check all have same length
                skipped_alignment_issue +=1
                continue

            combined_df = pd.concat(aligned_sensor_data, axis=1)
            combined_df.columns = ALL_FEATURE_COLUMNS # Ensure consistent column names

        except Exception as e:
            # print(f"Error during alignment for trial {trial_key}: {e}")
            skipped_alignment_issue +=1
            continue
            
        if combined_df.empty or len(combined_df) < SEQUENCE_LENGTH:
            skipped_too_short_after_processing += 1
            continue
            
        processed_trials_data.append(combined_df.values)
        category = trial_metadata_map[trial_key]["category"]
        label = 1 if category == "FALLS" else 0
        labels.append(label)

    print(f"Skipped {skipped_missing_sensors} trials due to missing one or more sensor types.")
    print(f"Skipped {skipped_resampling_issue} trials due to resampling issues.")
    print(f"Skipped {skipped_alignment_issue} trials due to alignment issues (no common time range or length mismatch).")
    print(f"Skipped {skipped_too_short_after_processing} trials due to being too short after processing (length < {SEQUENCE_LENGTH}).")
    print(f"Successfully processed and combined sensor data for {len(processed_trials_data)} trials.")
    return processed_trials_data, labels


# --- 2. Create Sequences using Sliding Window ---\n",
def create_sequences(data_list, label_list, seq_length, step):
    X, y = [], []
    for i, trial_data in enumerate(data_list):
        trial_label = label_list[i]
        for j in range(0, len(trial_data) - seq_length + 1, step):
            sequence = trial_data[j:(j + seq_length)]
            X.append(sequence)
            y.append(trial_label)
    if not X: return np.array([]), np.array([]) # Handle case where no sequences are created
    return np.array(X), np.array(y)


# --- Main Script Execution ---
if not os.path.exists(DATASET_PATH):
    print(f"DATASET_PATH '{DATASET_PATH}' not found. Exiting.")
    # exit() # Uncomment if you want to strictly stop

trial_arrays, trial_labels = load_data_from_structured_folders(DATASET_PATH)

if not trial_arrays:
    print("No data loaded after processing. Exiting. Check dataset path, file contents, and resampling logic.")
else:
    print(f"\nLoaded {len(trial_arrays)} trials for sequence generation.")
    if trial_arrays:
        print(f"Example trial data shape (after resampling & alignment, before windowing): {trial_arrays[0].shape}")
        if trial_arrays[0].shape[1] != len(ALL_FEATURE_COLUMNS):
             print(f"WARNING: Expected {len(ALL_FEATURE_COLUMNS)} features, got {trial_arrays[0].shape[1]}.")
    print(f"Labels per trial: {len(trial_labels)}")
    if trial_labels: print(f"Label dist (per trial): {pd.Series(trial_labels).value_counts(normalize=True)}")

    X_sequences, y_sequences = create_sequences(trial_arrays, trial_labels, SEQUENCE_LENGTH, STEP)

    if X_sequences.size == 0: # Check if array is empty
        print("\nNo sequences created. Check trial lengths and SEQUENCE_LENGTH/STEP. Exiting.")
    else:
        print(f"\nCreated {X_sequences.shape[0]} sequences.")
        print(f"X_sequences shape: {X_sequences.shape}, y_sequences shape: {y_sequences.shape}")
        print(f"Label dist (per sequence): {pd.Series(y_sequences).value_counts(normalize=True)}")

        unique_labels, counts = np.unique(y_sequences, return_counts=True)
        # Ensure at least 2 samples per class for stratification if more than 1 class
        stratify_param = y_sequences if len(unique_labels) > 1 and all(c >= 2 for c in counts) else None


        X_train, X_test, y_train, y_test = train_test_split(
            X_sequences, y_sequences, test_size=0.25, random_state=42, stratify=stratify_param
        )
        print(f"\nTrain set: {X_train.shape}, {y_train.shape}; Test set: {X_test.shape}, {y_test.shape}")
        if y_train.size > 0: print(f"Train labels dist: {pd.Series(y_train).value_counts(normalize=True)}")
        if y_test.size > 0: print(f"Test labels dist: {pd.Series(y_test).value_counts(normalize=True)}")

        scaler = StandardScaler()
        # Reshape for scaling: (num_samples * sequence_length, num_features)
        X_train_reshaped = X_train.reshape(-1, X_train.shape[2])
        X_train_scaled_reshaped = scaler.fit_transform(X_train_reshaped)
        X_train = X_train_scaled_reshaped.reshape(X_train.shape) # Reshape back

        scaler_save_path = "/kaggle/working/scaler_50hz.gz"
        joblib.dump(scaler, scaler_save_path)
        print(f"Scaler saved to {scaler_save_path}")

        X_test_reshaped = X_test.reshape(-1, X_test.shape[2])
        X_test_scaled_reshaped = scaler.transform(X_test_reshaped)
        X_test = X_test_scaled_reshaped.reshape(X_test.shape)
        print("\nData scaled.")

        # --- Model Definition, Compilation, Training, Evaluation ---
        model = Sequential([
            
            Conv1D(filters=64, kernel_size=10, activation='relu',
            input_shape=(SEQUENCE_LENGTH, X_train.shape[2])),
            MaxPooling1D(pool_size=2),
            Dropout(0.3),
            Bidirectional(LSTM(64, return_sequences=True)), 
            Dropout(0.4),
            Bidirectional(LSTM(32, return_sequences=False)),
            Dropout(0.4),
            Dense(32, activation='relu'),
            Dense(1, activation='sigmoid')
        ])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        model.summary()

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1) # Adjusted min_lr
        ]

        class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        class_weight_dict = dict(enumerate(class_weights))
        print(f"Calculated Class weights: {class_weight_dict}")
        print("\nStarting model training...")
        history = model.fit(
            X_train, y_train, epochs=50, batch_size=64, validation_split=0.2,
            callbacks=callbacks, verbose=1
        )

        print("\nEvaluating model on the test set...")
        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

    
        y_pred_probs = model.predict(X_test)
        y_pred = (y_pred_probs > 0.8).astype(int).flatten()
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6,5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['ADL (0)', 'Fall (1)'], yticklabels=['ADL (0)', 'Fall (1)'])
        plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix'); plt.show()
        print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['ADL (0)', 'Fall (1)']))

        model_save_path = "/kaggle/working/fall_detection_rnn_model_50hz.keras"
        model.save(model_save_path)
        print(f"\nModel saved to {model_save_path}")


In [ ]:
from tensorflow.keras.models import load_model

model = load_model('/kaggle/working/fall_detection_rnn_model_50hz.keras')


In [ ]:
import numpy as np

# Example input: 1 sample, 100 timesteps, 6 features (e.g., accelerometer + gyroscope)
sample_input = np.random.rand(1, 100, 6)  # Replace with real data


In [ ]:
model.summary()


In [ ]:
import numpy as np 
import pandas as pd 
import os

import matplotlib.pyplot as plt
import csv
import itertools
import collections

import pywt
from scipy import stats

from sklearn.utils import resample
from sklearn.model_selection import train_test_split

import keras
from keras.models import Sequential, Model
from keras.layers import Conv1D, AvgPool1D, MaxPooling1D, Flatten, Dense, Dropout, Softmax, Concatenate
from keras.layers import BatchNormalization
from keras.optimizers import Adam 
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.utils import plot_model
from keras import regularizers



In [ ]:
plt.rcParams["figure.figsize"] = (30,6)
plt.rcParams['lines.linewidth'] = 1
plt.rcParams['lines.color'] = 'b'
plt.rcParams['axes.grid'] = True 


In [ ]:
def denoise(data): 
    w = pywt.Wavelet('sym4')
    maxlev = pywt.dwt_max_level(len(data), w.dec_len)
    threshold = 0.04 # Threshold for filtering

    coeffs = pywt.wavedec(data, 'sym4', level=maxlev)
    for i in range(1, len(coeffs)):
        coeffs[i] = pywt.threshold(coeffs[i], threshold*max(coeffs[i]))
        
    datarec = pywt.waverec(coeffs, 'sym4')
    
    return datarec

In [ ]:
path = '/kaggle/input/mitbih-database/'
window_size = 180
maximum_counting = 10000

classes = ['N', 'S', 'V', 'F', 'Q']
n_classes = len(classes)
count_classes = [0]*n_classes

X = list()
y = list()

In [ ]:
filenames = next(os.walk(path))[2]

# Split and save .csv , .txt 
records = list()
annotations = list()
filenames.sort()

In [ ]:
for f in filenames:
    filename, file_extension = os.path.splitext(f)
    
    # *.csv
    if(file_extension == '.csv'):
        records.append(path + filename + file_extension)

    # *.txt
    else:
        annotations.append(path + filename + file_extension)

In [ ]:
for r in range(0,len(records)):
    signals = []

    with open(records[r], 'rt') as csvfile:
        spamreader = csv.reader(csvfile, delimiter=',', quotechar='|') # read CSV file\
        row_index = -1
        for row in spamreader:
            if(row_index >= 0):
                signals.insert(row_index, int(row[1]))
            row_index += 1
            
    # Plot an example to the signals
    if r == 1:
        # Plot each patient's signal
        plt.title(records[1] + " Wave")
        plt.plot(signals[0:700])
        plt.show()
        
    signals = denoise(signals)
    # Plot an example to the signals
    if r == 1:
        # Plot each patient's signal
        plt.title(records[1] + " wave after denoised")
        plt.plot(signals[0:700])
        plt.show()
        
    signals = stats.zscore(signals)
    # Plot an example to the signals
    if r == 1:
        # Plot each patient's signal
        plt.title(records[1] + " wave after z-score normalization ")
        plt.plot(signals[0:700])
        plt.show()
    
    # Read anotations: R position and Arrhythmia class
    example_beat_printed = False
    with open(annotations[r], 'r') as fileID:
        data = fileID.readlines() 
        beat = list()

        for d in range(1, len(data)): # 0 index is Chart Head
            splitted = data[d].split(' ')
            splitted = filter(None, splitted)
            next(splitted) # Time... Clipping
            pos = int(next(splitted)) # Sample ID
            arrhythmia_type = next(splitted) # Type
            if(arrhythmia_type in classes):
                arrhythmia_index = classes.index(arrhythmia_type)
#                 if count_classes[arrhythmia_index] > maximum_counting: # avoid overfitting
#                     pass
#                 else:
                count_classes[arrhythmia_index] += 1
                if(window_size <= pos and pos < (len(signals) - window_size)):
                    beat = signals[pos-window_size:pos+window_size]     ## REPLACE WITH R-PEAK DETECTION
                    # Plot an example to a beat    
                    if r == 1 and not example_beat_printed: 
                        plt.title("A Beat from " + records[1] + " Wave")
                        plt.plot(beat)
                        plt.show()
                        example_beat_printed = True

                    X.append(beat)
                    y.append(arrhythmia_index)

# data shape
print(np.shape(X), np.shape(y))


In [ ]:
for i in range(0,len(X)):
        X[i] = np.append(X[i], y[i])

print(np.shape(X))

In [ ]:
X_train_df = pd.DataFrame(X)
per_class = X_train_df[X_train_df.shape[1]-1].value_counts()
print(per_class)
plt.figure(figsize=(20,10))
my_circle=plt.Circle( (0,0), 0.7, color='white')
plt.pie(per_class, labels=['N', 'S', 'V', 'F', 'Q'], colors=['tab:blue','tab:orange','tab:purple','tab:olive','tab:green'],autopct='%1.1f%%')
p=plt.gcf()
p.gca().add_artist(my_circle)
plt.show()

In [ ]:
df_1=X_train_df[X_train_df[X_train_df.shape[1]-1]==1]
df_2=X_train_df[X_train_df[X_train_df.shape[1]-1]==2]
df_3=X_train_df[X_train_df[X_train_df.shape[1]-1]==3]
df_4=X_train_df[X_train_df[X_train_df.shape[1]-1]==4]
df_0=(X_train_df[X_train_df[X_train_df.shape[1]-1]==0]).sample(n=5000,random_state=42)

df_1_upsample=resample(df_1,replace=True,n_samples=5000,random_state=122)
df_2_upsample=resample(df_2,replace=True,n_samples=5000,random_state=123)
df_3_upsample=resample(df_3,replace=True,n_samples=5000,random_state=124)
df_4_upsample=resample(df_4,replace=True,n_samples=5000,random_state=125)

X_train_df=pd.concat([df_0,df_1_upsample,df_2_upsample,df_3_upsample,df_4_upsample])

In [ ]:
per_class = X_train_df[X_train_df.shape[1]-1].value_counts()
print(per_class)
plt.figure(figsize=(20,10))
my_circle=plt.Circle( (0,0), 0.7, color='white')
plt.pie(per_class, labels=['N', 'S', 'V', 'F', 'Q'], colors=['tab:blue','tab:orange','tab:purple','tab:olive','tab:green'],autopct='%1.1f%%')
p=plt.gcf()
p.gca().add_artist(my_circle)
plt.show()

In [ ]:
train, test = train_test_split(X_train_df, test_size=0.20)

print("X_train : ", np.shape(train))
print("X_test  : ", np.shape(test))

In [ ]:
target_train=train[train.shape[1]-1]
target_test=test[test.shape[1]-1]
train_y=to_categorical(target_train)
test_y=to_categorical(target_test)
print(np.shape(train_y), np.shape(test_y))

In [ ]:
model = Sequential()

model.add(Conv1D(filters=16, kernel_size=11, strides=1, padding='same' , activation = 'relu', input_shape=(360,1)))
model.add(MaxPooling1D(pool_size=5, strides=2))
model.add(Conv1D(filters=32, kernel_size=13, strides=1, padding='same' , activation = 'relu'))
model.add(MaxPooling1D(pool_size=5, strides=2))
model.add(Conv1D(filters=64, kernel_size=15, strides=1, padding='same' , activation = 'relu'))
model.add(MaxPooling1D(pool_size=5, strides=2))
model.add(Conv1D(filters=128, kernel_size=17, strides=1, padding='same' , activation = 'relu'))
model.add(MaxPooling1D(pool_size=5, strides=2))
model.add(Flatten())
model.add(Dropout(0.5))
model.add(Dense(100, activation='relu'))
model.add(Dense(50, activation='relu'))
model.add(Dense(5, activation='softmax'))
model.summary()
model.compile(loss='categorical_crossentropy', optimizer=Adam(), metrics=['accuracy'])
history = model.fit(train_x, train_y, batch_size=36, epochs=60, verbose=1, validation_data=(test_x, test_y))

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
model.predict(test_x)

In [ ]:
score = model.evaluate(test_x, test_y)

print('Test Loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score

import seaborn as sns
from sklearn.metrics import confusion_matrix

y_true=[]
for element in test_y:
    y_true.append(np.argmax(element))
prediction_proba=model.predict(test_x)
prediction=np.argmax(prediction_proba,axis=1)
ax=plt.subplot()
custCnnConfMat = confusion_matrix(y_true, prediction)
sns.heatmap(custCnnConfMat, annot=True,fmt='d',ax=ax)
ax.set_title('Confusion Matrix'); 
ax.xaxis.set_ticklabels(classes); ax.yaxis.set_ticklabels(classes);
plt.savefig('cm')
model_save_path = "/kaggle/working/arrhythmia_classification_model.keras"
model.save(model_save_path)
print(f"\nModel saved to {model_save_path}")


In [ ]:
from sklearn.metrics import classification_report

cf = classification_report(y_true, prediction, target_names=classes,digits=4)
print(cf)